Generate Delphi-panel responses, retaining round-level decisions, final answers and token usage.


In [1]:
from pathlib import Path
import os
_REPOSITORY_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(_REPOSITORY_ROOT).startswith('\\\\?\\')):
    _REPOSITORY_ROOT = Path('\\\\?\\' + str(_REPOSITORY_ROOT))
_NOTEBOOK_DIR = _REPOSITORY_ROOT / 'experiments/external_benchmarks/initial_runs/ensemble_comparisons'
os.chdir(_NOTEBOOK_DIR)


In [2]:
import collections
import json
import os
from typing import Dict, List, Optional, Sequence
import numpy as np
import pandas as pd
from scipy import stats
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
BASELINE_DIRS = {'gpt': 'baseline_gpt', 'qwen': 'baseline_qwen', 'qwen36': 'baseline_qwen36', 'alice': 'baseline_alice', 'deepseekv4': 'baseline_deepseekv4'}
DELPHI_DIRS = {'Optimum': 'heterogeneous_gpt120_organizer', 'AC-Alice': 'panel_gpt120_organizer_qwen_alice', 'AC-DeepSeek': 'panel_gpt120_organizer_qwen_deepseek', 'DeepSeekTrio': 'three_deepseek_members'}
for registry in [BASELINE_DIRS, DELPHI_DIRS]:
    for system, folder in list(registry.items()):
        local = _REPOSITORY_ROOT / 'experiments/external_benchmarks/initial_runs' / folder
        relocated = _REPOSITORY_ROOT / 'experiments/external_benchmarks/medmcqa' / folder
        if local.is_dir():
            registry[system] = str(local)
        elif relocated.is_dir():
            registry[system] = str(relocated)
        else:
            raise FileNotFoundError(folder)
PANEL_MEMBERS = {'Optimum': ['gpt', 'qwen', 'qwen36'], 'AC-Alice': ['gpt', 'qwen', 'alice'], 'AC-DeepSeek': ['gpt', 'qwen', 'deepseekv4'], 'DeepSeekTrio': None}
VALID_OPTIONS = frozenset('ABCD')
TIE_SEED = 42
ALPHA = 0.05
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)
print('ROOT:', ROOT)


ROOT: .\experiments\external_benchmarks\initial_runs


In [3]:
def read_run(path: str) -> pd.DataFrame:
    df = pd.read_csv(os.path.abspath(path), sep=';', encoding='utf-8-sig', low_memory=False)
    df = df[df['case_id'].notna()].drop_duplicates('case_id')
    return df.reset_index(drop=True)

def letters(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.upper().str[:1]

def as_indexed(df: pd.DataFrame, column: str) -> pd.Series:
    return pd.Series(letters(df[column]).values, index=df['case_id'].values)

def majority_vote(votes: Sequence[str], rule: str, rng: np.random.Generator) -> Optional[str]:
    counts = collections.Counter((v for v in votes if v in VALID_OPTIONS))
    if not counts:
        return None
    top = max(counts.values())
    leaders = sorted((k for k, v in counts.items() if v == top))
    if len(leaders) == 1:
        return leaders[0]
    if rule == 'random':
        return leaders[int(rng.integers(len(leaders)))]
    if rule == 'alpha':
        return leaders[0]
    if rule == 'drop':
        return None
    raise ValueError(f'unknown tie rule: {rule}')

def vote_frame(frame: pd.DataFrame, rule: str) -> pd.Series:
    rng = np.random.default_rng(TIE_SEED)
    return frame.apply(lambda row: majority_vote(list(row.values), rule, rng), axis=1)

def mcnemar(a_correct: pd.Series, b_correct: pd.Series) -> Dict[str, float]:
    n10 = int((a_correct & ~b_correct).sum())
    n01 = int((~a_correct & b_correct).sum())
    p = stats.binomtest(n10, n10 + n01, 0.5).pvalue if n10 + n01 else 1.0
    return {'n': int(len(a_correct)), 'acc_a': float(a_correct.mean()), 'acc_b': float(b_correct.mean()), 'delta': float(a_correct.mean() - b_correct.mean()), 'n10': n10, 'n01': n01, 'p': float(p)}

def holm(pvalues: List[float]) -> List[float]:
    m = len(pvalues)
    order = np.argsort(pvalues)
    adjusted = np.empty(m, dtype=float)
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * pvalues[idx])
        adjusted[idx] = min(1.0, running)
    return adjusted.tolist()


In [4]:
truth_df = read_run(os.path.join(ROOT, BASELINE_DIRS['gpt'], 'baseline_medmcqa_gpt.csv'))
TRUTH = as_indexed(truth_df, 'correct_option_letter')
BASELINE: Dict[str, pd.Series] = {}
for name, folder in BASELINE_DIRS.items():
    df = read_run(os.path.join(ROOT, folder, f'baseline_medmcqa_{name}.csv'))
    BASELINE[name] = as_indexed(df, 'baseline_selected_option')
DELPHI: Dict[str, pd.Series] = {}
R1_VOTES: Dict[str, pd.DataFrame] = {}
ROUNDS: Dict[str, pd.Series] = {}
for name, folder in DELPHI_DIRS.items():
    df = read_run(os.path.join(ROOT, folder, 'delphi_medmcqa_noroles.csv'))
    idx = df['case_id'].values
    DELPHI[name] = as_indexed(df, 'delphi_consensus_selected_option')
    r1_cols = [c for c in df.columns if c.startswith('r1_selected_option__')]
    R1_VOTES[name] = pd.DataFrame({c.split('__', 1)[1]: letters(df[c]).values for c in r1_cols}, index=idx)
    ROUNDS[name] = pd.Series(pd.to_numeric(df['delphi_rounds_executed'], errors='coerce').values, index=idx)
rows = []
for name, s in BASELINE.items():
    ok = s == TRUTH.reindex(s.index)
    rows.append(('baseline', name, len(s), int((~s.isin(VALID_OPTIONS)).sum()), ok.mean()))
for name, s in DELPHI.items():
    ok = s == TRUTH.reindex(s.index)
    rows.append(('delphi', name, len(s), int((~s.isin(VALID_OPTIONS)).sum()), ok.mean()))
sanity = pd.DataFrame(rows, columns=['тип', 'система', 'кейсов', 'невалидных', 'accuracy'])
print(sanity.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
print()
for name in DELPHI_DIRS:
    dist = ROUNDS[name].value_counts().sort_index().to_dict()
    print(f'{name:<14} раундов выполнено: {dist}')


     тип      система  кейсов  невалидных  accuracy
baseline          gpt    1000           3    0.8130
baseline         qwen    1000          11    0.8140
baseline       qwen36    1000          20    0.8500
baseline        alice    1000           9    0.8160
baseline   deepseekv4    1000           5    0.8790
  delphi      Optimum    1000           0    0.8760
  delphi     AC-Alice    1000           0    0.8620
  delphi  AC-DeepSeek    1000           0    0.8810
  delphi DeepSeekTrio    1000           0    0.8930

Optimum        раундов выполнено: {1.0: 821, 2.0: 129, 3.0: 50}
AC-Alice       раундов выполнено: {1.0: 801, 2.0: 124, 3.0: 75}
AC-DeepSeek    раундов выполнено: {1.0: 827, 2.0: 108, 3.0: 65}
DeepSeekTrio   раундов выполнено: {1.0: 918, 2.0: 55, 3.0: 27}


In [5]:
def panel_frame(members: List[str], index) -> pd.DataFrame:
    return pd.DataFrame({f'{m}#{i}': BASELINE[m].reindex(index) for i, m in enumerate(members)})

def common_index(*series_like) -> pd.Index:
    idx = None
    for s in series_like:
        cur = pd.Index(s.index)
        idx = cur if idx is None else idx.intersection(cur)
    return idx
print('Структура голосов трёх baseline-прогонов\n')
structure_rows = []
for panel, members in PANEL_MEMBERS.items():
    if members is None:
        continue
    idx = common_index(TRUTH, DELPHI[panel], *[BASELINE[m] for m in members])
    votes = panel_frame(members, idx)

    def pattern(row):
        counts = collections.Counter((v for v in row.values if v in VALID_OPTIONS))
        if not counts:
            return 'нет валидных'
        top = max(counts.values())
        leaders = [k for k, v in counts.items() if v == top]
        if top == 3:
            return 'единогласно 3-0'
        if top == 2:
            return 'большинство 2-1'
        return 'ничья' if len(leaders) > 1 else 'один голос'
    dist = votes.apply(pattern, axis=1).value_counts().to_dict()
    structure_rows.append({'панель': panel, 'n': len(idx), **dist})
print(pd.DataFrame(structure_rows).fillna(0).to_string(index=False))


Структура голосов трёх baseline-прогонов



     панель    n  единогласно 3-0  большинство 2-1  ничья  один голос  нет валидных
    Optimum 1000              741              228     28           3           0.0
   AC-Alice 1000              745              231     20           3           1.0
AC-DeepSeek 1000              743              234     22           1           0.0


In [6]:
exp1_rows = []
for panel, members in PANEL_MEMBERS.items():
    if members is None:
        print(f'{panel}: пропущен - нет трёх независимых baseline-прогонов (гомогенная панель)')
        continue
    idx = common_index(TRUTH, DELPHI[panel], *[BASELINE[m] for m in members])
    truth = TRUTH.reindex(idx)
    delphi = DELPHI[panel].reindex(idx)
    votes = panel_frame(members, idx)
    for rule in ('random', 'drop', 'alpha'):
        mv = vote_frame(votes, rule)
        keep = mv.notna()
        res = mcnemar(delphi[keep] == truth[keep], mv[keep] == truth[keep])
        exp1_rows.append({'панель': panel, 'состав': '+'.join(members), 'правило ничьей': rule, 'n': res['n'], 'Delphi': res['acc_a'], 'majority': res['acc_b'], 'delta': res['delta'], 'disc D/M': f"{res['n10']}/{res['n01']}", 'p': res['p']})
exp1 = pd.DataFrame(exp1_rows)
main1 = exp1['правило ничьей'] == 'random'
exp1.loc[main1, 'p_holm'] = holm(exp1.loc[main1, 'p'].tolist())
print(exp1.to_string(index=False, float_format=lambda v: f'{v:.4f}'))


DeepSeekTrio: пропущен - нет трёх независимых baseline-прогонов (гомогенная панель)
     панель              состав правило ничьей    n  Delphi  majority  delta disc D/M      p  p_holm
    Optimum     gpt+qwen+qwen36         random 1000  0.8760    0.8640 0.0120    31/19 0.1189  0.2378
    Optimum     gpt+qwen+qwen36           drop  972  0.8879    0.8776 0.0103    25/15 0.1539     NaN
    Optimum     gpt+qwen+qwen36          alpha 1000  0.8760    0.8590 0.0170    33/16 0.0213     NaN
   AC-Alice      gpt+qwen+alice         random  999  0.8629    0.8428 0.0200    45/25 0.0225  0.0674
   AC-Alice      gpt+qwen+alice           drop  979  0.8713    0.8519 0.0194    40/21 0.0204     NaN
   AC-Alice      gpt+qwen+alice          alpha  999  0.8629    0.8388 0.0240    46/22 0.0049     NaN
AC-DeepSeek gpt+qwen+deepseekv4         random 1000  0.8810    0.8700 0.0110    33/22 0.1770  0.2378
AC-DeepSeek gpt+qwen+deepseekv4           drop  978  0.8926    0.8845 0.0082    27/19 0.3020     NaN
AC-Deep

In [7]:
exp2_rows = []
missed_breakdown = []
for panel, members in PANEL_MEMBERS.items():
    if members is None:
        votes_src = R1_VOTES[panel]
        idx = common_index(TRUTH, DELPHI[panel], votes_src)
        votes = votes_src.reindex(idx)
        source = 'голоса раунда 1'
    else:
        idx = common_index(TRUTH, DELPHI[panel], *[BASELINE[m] for m in members])
        votes = panel_frame(members, idx)
        source = 'baseline-прогоны'
    truth = TRUTH.reindex(idx)
    delphi_ok = DELPHI[panel].reindex(idx) == truth
    hits = votes.eq(truth, axis=0)
    n_right = hits.sum(axis=1)
    oracle_ok = n_right > 0
    extracted = float(delphi_ok[oracle_ok].mean())
    missed = float((oracle_ok & ~delphi_ok).mean())
    exp2_rows.append({'панель': panel, 'источник голосов': source, 'n': len(idx), 'Delphi': float(delphi_ok.mean()), 'oracle': float(oracle_ok.mean()), 'разрыв': float(oracle_ok.mean() - delphi_ok.mean()), 'извлечено': extracted, 'упущено': missed})
    miss = oracle_ok & ~delphi_ok
    row = {'панель': panel, 'упущено всего': int(miss.sum())}
    for k in (1, 2, 3):
        row[f'знали {k}'] = int((miss & (n_right == k)).sum())
    missed_breakdown.append(row)
exp2 = pd.DataFrame(exp2_rows)
print(exp2.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
print()
for row in exp2_rows:
    if row['источник голосов'] == 'голоса раунда 1' and row['разрыв'] < 0:
        print()
        print(f"  ВНИМАНИЕ: у панели {row['панель']} oracle посчитан по голосам раунда 1 и не ограничивает Delphi сверху (см. оговорку выше).")
print()
print('Упущенные кейсы по числу участников, знавших правильный ответ:')
print(pd.DataFrame(missed_breakdown).to_string(index=False))


      панель источник голосов    n  Delphi  oracle  разрыв  извлечено  упущено
     Optimum baseline-прогоны 1000  0.8760  0.9260  0.0500     0.9374   0.0580
    AC-Alice baseline-прогоны 1000  0.8620  0.9140  0.0520     0.9300   0.0640
 AC-DeepSeek baseline-прогоны 1000  0.8810  0.9330  0.0520     0.9368   0.0590
DeepSeekTrio  голоса раунда 1 1000  0.8930  0.8810 -0.0120     0.9875   0.0110


  ВНИМАНИЕ: у панели DeepSeekTrio oracle посчитан по голосам раунда 1 и не ограничивает Delphi сверху (см. оговорку выше).

Упущенные кейсы по числу участников, знавших правильный ответ:
      панель  упущено всего  знали 1  знали 2  знали 3
     Optimum             58       43       12        3
    AC-Alice             64       43       19        2
 AC-DeepSeek             59       40       17        2
DeepSeekTrio             11        7        4        0


In [8]:
exp3_rows = []
for panel in DELPHI_DIRS:
    idx = common_index(TRUTH, DELPHI[panel], R1_VOTES[panel], ROUNDS[panel])
    truth = TRUTH.reindex(idx)
    delphi = DELPHI[panel].reindex(idx)
    rounds = ROUNDS[panel].reindex(idx)
    mv = vote_frame(R1_VOTES[panel].reindex(idx), 'random')
    identical = int((delphi == mv).sum())
    for label, mask in (('вся выборка', pd.Series(True, index=idx)), ('раунд 2+ (была делиберация)', rounds >= 2), ('остановлено после раунда 1', rounds == 1)):
        keep = mask & mv.notna() & truth.notna()
        if not keep.any():
            continue
        res = mcnemar(delphi[keep] == truth[keep], mv[keep] == truth[keep])
        exp3_rows.append({'панель': panel, 'срез': label, 'n': res['n'], 'Delphi': res['acc_a'], 'R1-vote': res['acc_b'], 'delta': res['delta'], 'disc D/R1': f"{res['n10']}/{res['n01']}", 'p': res['p'], 'ответ совпал': identical})
exp3 = pd.DataFrame(exp3_rows)
main3 = exp3['срез'] == 'раунд 2+ (была делиберация)'
exp3.loc[main3, 'p_holm'] = holm(exp3.loc[main3, 'p'].tolist())
print(exp3.to_string(index=False, float_format=lambda v: f'{v:.4f}'))


      панель                        срез    n  Delphi  R1-vote  delta disc D/R1      p  ответ совпал  p_holm
     Optimum                 вся выборка 1000  0.8760   0.8660 0.0100      19/9 0.0872           969     NaN
     Optimum раунд 2+ (была делиберация)  179  0.6536   0.5978 0.0559      19/9 0.0872           969  0.2615
     Optimum  остановлено после раунда 1  821  0.9245   0.9245 0.0000       0/0 1.0000           969     NaN
    AC-Alice                 вся выборка 1000  0.8620   0.8500 0.0120     22/10 0.0501           961     NaN
    AC-Alice раунд 2+ (была делиберация)  199  0.5729   0.5126 0.0603     22/10 0.0501           961  0.2004
    AC-Alice  остановлено после раунда 1  801  0.9338   0.9338 0.0000       0/0 1.0000           961     NaN
 AC-DeepSeek                 вся выборка 1000  0.8810   0.8730 0.0080     21/13 0.2295           955     NaN
 AC-DeepSeek раунд 2+ (была делиберация)  173  0.6590   0.6127 0.0462     21/13 0.2295           955  0.4590
 AC-DeepSeek  остан

In [9]:
ladder_rows = []
for panel, members in PANEL_MEMBERS.items():
    if members is None:
        continue
    idx = common_index(TRUTH, DELPHI[panel], R1_VOTES[panel], *[BASELINE[m] for m in members])
    truth = TRUTH.reindex(idx)
    votes = panel_frame(members, idx)
    hits = votes.eq(truth, axis=0)
    ladder_rows.append({'панель': panel, 'n': len(idx), 'лучший участник': max((float((BASELINE[m].reindex(idx) == truth).mean()) for m in members)), 'majority baseline': float((vote_frame(votes, 'random')[idx] == truth).mean()), 'R1-vote панели': float((vote_frame(R1_VOTES[panel].reindex(idx), 'random') == truth).mean()), 'Delphi (3 раунда)': float((DELPHI[panel].reindex(idx) == truth).mean()), 'oracle': float((hits.sum(axis=1) > 0).mean())})
ladder = pd.DataFrame(ladder_rows)
best_overall = max(((s == TRUTH.reindex(s.index)).mean() for s in BASELINE.values()))
print(ladder.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
print()
print(f'Для справки: лучшая одиночная модель во всей папке = {best_overall:.4f}')
print()
print("Читать так: прирост между 'majority baseline' и 'R1-vote панели' даёт промпт")
print("раунда 1 (декомпозиция на подвопросы); прирост между 'R1-vote панели' и")
print("'Delphi (3 раунда)' даёт делиберация; 'oracle' - потолок для этой панели.")


     панель    n  лучший участник  majority baseline  R1-vote панели  Delphi (3 раунда)  oracle
    Optimum 1000           0.8500             0.8640          0.8660             0.8760  0.9260
   AC-Alice 1000           0.8160             0.8420          0.8500             0.8620  0.9140
AC-DeepSeek 1000           0.8790             0.8700          0.8730             0.8810  0.9330

Для справки: лучшая одиночная модель во всей папке = 0.8790

Читать так: прирост между 'majority baseline' и 'R1-vote панели' даёт промпт
раунда 1 (декомпозиция на подвопросы); прирост между 'R1-vote панели' и
'Delphi (3 раунда)' даёт делиберация; 'oracle' - потолок для этой панели.


In [10]:
summary = {'dataset': 'MedMCQA', 'n_cases': int(len(TRUTH)), 'tie_rule': 'random', 'tie_seed': TIE_SEED, 'exp1_delphi_vs_baseline_majority': json.loads(exp1[exp1['правило ничьей'] == 'random'].to_json(orient='records', force_ascii=False)), 'exp2_delphi_vs_oracle': json.loads(exp2.to_json(orient='records', force_ascii=False)), 'exp3_delphi_vs_r1_vote': json.loads(exp3.to_json(orient='records', force_ascii=False)), 'ladder': json.loads(ladder.to_json(orient='records', force_ascii=False))}
out = os.path.abspath('analysis_external_results.json')
with open(out, 'w', encoding='utf-8') as fh:
    json.dump(summary, fh, ensure_ascii=False, indent=2)
print('saved:', out)


saved: .\experiments\external_benchmarks\initial_runs\ensemble_comparisons\analysis_external_results.json
